In [96]:
import numpy as np
from scipy.special import lpmv
from math import factorial

Building the spherical harmonic basis. Given a pair of integers(l, m) and a point (theta, phi) on the sphere, it returns the value of that particular spherical harmonic at that point.

In [99]:
def real_Y(l, m, theta, phi):
    am = abs(m)
    norm = np.sqrt((2 * l +1) / (4 * np.pi) * factorial(l - am) / factorial(l + am))
    P = lpmv(am, l, np.cos(theta))
    if m ==0:
        return norm * P
    elif m > 0:
        return np.sqrt(2) * norm * P * np.cos(am * phi)
    else:
        return np.sqrt(2) * norm * P * np.sin(am * phi)

The call function evaluates the spherical harmonics at different thetas and phis. The eval cart function takes our (x,y,z) vector, normalizes it, and the converts it into a (theta, phi) vector to be evaluated. This let's us treat f as a function on all of R^3 to allows to take an ordinary gradient. 

In [102]:
class SphericalFunction:
    def __init__(self, coeffs):
        self.coeffs = coeffs

    def __call__(self, theta, phi):
        val = 0.0 
        for (l, m), c in self.coeffs.items():
            val += c * real_Y(l, m, theta, phi)
        return val

    def eval_cart(self, v):
        v = np.asarray(v, dtype=float)
        r = np.linalg.norm(v)
        vhat = v/r
        theta = np.arccos(np.clip(vhat[2], -1.0, 1.0))
        phi = np.arctan2(vhat[1], vhat[0]) % (2 * np.pi)
        return self(theta, phi)
    

For any point on the sphere the plane tangent is 2d, so here we build the orthonormal basis(e1,e2). 

In [105]:
def tangent_basis(v):
    helper = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(helper, v)) > 0.9:
        helper = np.array([0.0, 1.0, 0.0])
    e1 = np.cross(v, helper)
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(v, e1)
    e2 /= np.linalg.norm(e2)
    return e1, e2

We then take take the gradient.

In [ ]:
def euclidean_gradient(F, v, h=1e-5):
    grad = np.zeros(3)
    for i in range(3):
        dv = np.zeros(3)
        dv[i] = h
        grad[i] = (F.eval_cart(v + dv) - F.eval_cart(v - dv)) / (2 * h)
    return grad

In [138]:
def tangential_gradient(F, v, h=1e5):
    v = v / np.linalg.norm(v)
    g = euclidean_gradient(F, v, h)
    return g - np.dot(g, v) * v

Here we build the hessian which is a matrix of partial derivatives of f, and allows us to classify critical points, all partial derivatives must be 0. 

In [141]:
def tangential_hessian(F, v, h=1e3):
    v = v / np.linalg.norm(v)
    e1, e2 = tangent_basis(v)

    def f_local(a, b):
        w = v + a * e1 + b * e2
        return F.eval_cart(w)

    f0 = f_local(0,0)
    fpp = f_local(h, 0); fmm = f_local(-h, 0)
    fqq = f_local(0, h); fnn = f_local(0, -h)
    fpq = f_local(h,h); fmn = f_local(-h, -h)
    fpn = f_local(h, -h); fmq = f_local(-h,h)

    d2f_da2 = (fpp - 2 * f0 + fmn) / h**2
    d2f_db2 = (fqq -2 *f0 +fnn) / h**2
    d2f_dadb = (fpq - fpn - fmq +fmn) / (4 * h**2)

    return np.array([[d2f_da2, d2f_dadb], [d2f_dadb, d2f_db2]])

Using Newton's method to find roots. 

In [144]:
def find_critical_point(F, v0, tol=1e-10, max_iter=100, damping=1.0):
    v = np.array(v0, dtype=float)
    v /= np.linalg.norm(v)

    for _ in range(max_iter):
        e1, e2 = tangent_basis(v)
        g = tangential_gradient(F, v)
        g_local = np.array([np.dot(g, e1), np.dot(g, e2)])
        if np.linalg.norm(g_local) < tol:
            break

        H = tangential_hessian(F, v)
        try:
            delta = np.linalg.solve(H, -g_local)
        except np.linalg.LinAlgError:
            delta = -g_local

        step_norm = np.linalg.norm(delta)
        if step_norm > 0.5:
            delta *= 0.5 / step_norm
        v = v + damping * (delta[0] * e1 + delta[1] * e2)
        v /= np.linalg.norm(v)

    return v, np.linalg.norm(tangential_gradient(F, v))

Defining a grid of points to start Newton's method on since Newton's method only converges to the critical point closest to the initial point. 

In [147]:
def fibonacci_sphere(n):
    points = []
    phi = np.pi * (3.0 - np.sqrt(5.0))
    for i in range(n):
        y = 1 - (i / float(n-1)) * 2
        radius = np.sqrt(1 - y *y)
        theta_f = phi * i
        x = np.cos(theta_f) * radius
        z = np.sin(theta_f) * radius
        points.append([x, y, z])
    return np.array(points)
    

In [154]:
def find_all_critical_points(F, n_starts=200, dedupe_tol=1e-3, grad_tol=1e-6):
    starts = fibonacci_sphere(n_starts)
    found = []
    for v0 in starts:
        v, gnorm = find_critical_point(F, v0)
        if gnorm > grad_tol:
            continue
        is_new = True
        for u in found:
            if np.linalg.norm(v - u) < dedupe_tol or np.linalg.norm(v +u) < dedupe_tol:
                is_new = False
                break
        if is_new:
            found.append(v)
    results = []
    for v in found:
        H = tangential_hessian(F, v)
        eigs = np.linalg.eigvalsh(H)
        if np.all(eigs > -1e-8):
            kind = "minimum"
        elif np.all(eigs < -1e-8):
            kind = "maximum"
        else:
            kind = "saddle"
        theta = np.arccos(np.clip(v[2], -1, 1))
        phi = np.arctan2(v[1], v[0]) % (2 * np.pi)
        results.append({
            "xyz": v, "theta": theta, "phi": phi,
            "value": F.eval_cart(v), "type": kind, "hessian_eigs": eigs,
        })
    return results

In [156]:
coeffs = {(1, 0): 1.0, (1, 1): 0.5, (2, 0): 0.8, (2, 2): -0.3}  # or your fitted c_lm
F = SphericalFunction(coeffs)
crit_pts = find_all_critical_points(F, n_starts=300)